### Funciones para hacer consulta

## 📊 Tipos de Propiedades Soportadas

El scraper maneja **TODOS** los tipos de propiedades disponibles en Lamudi y los clasifica automáticamente:

### **Residenciales (con habitaciones/baños)**
- **Casa**: Casa, Casa en Fraccionamiento, Villa, Casa en Condominio, Quinta/Hacienda
- **Departamento**: Departamento, Estudio, Dúplex, Loft, Penthouse, Condominio Horizontal

### **Comercial (sin habitaciones)**
- Local en Centro Comercial
- Tienda/Local Comercial
- Nave Industrial/Bodega
- Lote Comercial
- Entretenimiento/Hotel

### **Terreno (sin habitaciones)**
- Lote/Terreno
- Rancho/Huerta

### **Oficina (sin habitaciones)**
- Oficina
- Consultorio Médico
- Edificio

**Notas importantes:**
- ✅ Campos **SIEMPRE** presentes: `titulo`, `direccion`, `precio`, `superficie`, `tipo_vivienda`, `categoria`, `url`
- ⚠️ Campos opcionales (None para no-residenciales): `habitaciones`, `banios`, `planta`
- 📐 `superficie_terreno`: presente solo en casas (terrenos separados de construcción)
- 🏢 Amenidades (`alberca`, `gimnasio`, etc.): presentes solo en residenciales/comerciales con servicios


In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
from pathlib import Path
import os
import numpy as np
from unidecode import unidecode
from shapely.geometry import Point
import geopandas as gpd
import re
from datetime import datetime, timedelta
from dateutil import parser
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

def scrape_lamudi(start_url, output_filename):
    # Configuración de Chrome
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # Ejecutar en segundo plano
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    # Inicializar el driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.set_page_load_timeout(4)  # 4s: páginas cargan rápido
    driver.implicitly_wait(1)  # 1s: mínimo necesario
    
    # Lista para almacenar los datos
    data_list = []
    wait = WebDriverWait(driver, 3)  # 3s: suficiente para elementos
    total_propiedades = 0
    total_paginas = 0
    paginas_procesadas = 0
    
    # Cargar títulos existentes si el archivo existe
    titulos_existentes = set()
    if os.path.exists(output_filename):
        try:
            df_existente = pd.read_csv(output_filename)
            titulos_existentes = set(df_existente['titulo'].dropna().unique())
            print(f"📂 {len(titulos_existentes)} propiedades ya descargadas")
        except:
            print("⚠ Comenzando desde cero")
    
    columnas = [
        "titulo", "direccion", "tipo", "tipo_vivienda", "categoria", "precio", "superficie", "superficie_terreno",
        "habitaciones", "banios", "caracteristica_propiedad", "amenidades", "caracteristicas", 
        "planta", "descripcion", "url", "script_content", "fecha_publicacion", "fecha_consulta", "url_imagen"
    ]
    
    propiedades_nuevas = 0
    propiedades_repetidas = 0
    propiedades_con_error = 0

    try:
        driver.get(start_url)
        # SIN sleep aquí - driver.get ya espera el page_load_timeout

        try:
            total_text = driver.find_element(By.CSS_SELECTOR, "span[data-test='title-section-result-number']").text
            total_text = total_text.replace(',', '').strip()
            total_propiedades = int(total_text)
            total_paginas = (total_propiedades + 29) // 30
            print(f"📊 Total: {total_propiedades:,} | Páginas estimadas: {total_paginas}")
        except:
            print("⚠ No se pudo obtener total")
            total_propiedades = float('inf')
            total_paginas = float('inf')

        page_number = 1
        usar_boton = False  # Flag para intentar con botón si falla el método de página

        while True:
            paginas_procesadas += 1
            
            if paginas_procesadas > total_paginas:
                print(f"✓ Límite de páginas alcanzado ({total_paginas})")
                break

            # Estrategia 1: Intentar con ?page=N
            if not usar_boton:
                try:
                    # Construir URL con parámetro ?page=N
                    if "?" in start_url:
                        page_url = start_url + f"&page={page_number}"
                    else:
                        page_url = start_url + f"?page={page_number}"
                    
                    print(f"\n🌐 P{paginas_procesadas}: Cargando...")
                    driver.get(page_url)
                    print(f"   ✅ Página cargada")
                    page_number += 1
                    
                except Exception as e:
                    print(f"⚠ Método ?page= falló. Intentando con botón...")
                    usar_boton = True
                    continue

            # Estrategia 2: Usar botón de siguiente página (fallback)
            if usar_boton:
                try:
                    next_button = driver.find_element(By.CSS_SELECTOR, "a#pagination-next")
                    next_url = next_button.get_attribute("href")
                    if next_url:
                        print(f"\n🌐 P{paginas_procesadas}: Siguiendo botón...")
                        driver.get(next_url)
                        print(f"   ✅ Página cargada")
                    else:
                        print(f"✓ Sin página siguiente. Fin en página {paginas_procesadas}")
                        break
                except NoSuchElementException:
                    # No hay botón de siguiente página = última página
                    print(f"✓ Última página alcanzada ({paginas_procesadas})")
                    break
                except Exception as e:
                    print(f"⚠ Error en navegación: {str(e)[:50]}. Finalizando...")
                    break

            try:
                # Esperar enlaces - máximo 3 segundos
                print(f"   🔍 Buscando propiedades...")
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".snippet__content a[target]")))
                
                links = driver.find_elements(By.CSS_SELECTOR, ".snippet__content a[target]")
                print(f"   📌 Links DOM encontrados: {len(links)}")
                
                property_links = [link.get_attribute("href") for link in links if link.get_attribute("href")]
                print(f"   ✔️ Links válidos (con href): {len(property_links)}")

                if not property_links:
                    print(f"✓ Sin propiedades. Fin en página {paginas_procesadas}")
                    break

                print(f"   📋 Procesando {len(property_links)} propiedades...")

                propiedades_nuevas_pagina = 0
                propiedades_repetidas_pagina = 0
                propiedades_error_pagina = 0

                for idx, property_url in enumerate(property_links):
                    try:
                        driver.get(property_url)
                        # SIN sleep - el get ya espera

                        propiedad = {}

                        try:
                            propiedad['titulo'] = driver.find_element(By.TAG_NAME, "h1").text
                        except:
                            propiedad['titulo'] = None
                        
                        if propiedad['titulo'] and propiedad['titulo'] in titulos_existentes:
                            propiedades_repetidas += 1
                            propiedades_repetidas_pagina += 1
                            print(f"      ⚠️  {idx+1}/30 REPETIDA: {propiedad['titulo'][:35]}...")
                            continue
                        
                        try:
                            propiedad['direccion'] = driver.find_element(By.CSS_SELECTOR, "div.location-map__location-address-map").text
                        except:
                            propiedad['direccion'] = None
                        try:
                            propiedad['tipo'] = driver.find_element(By.CSS_SELECTOR, ".property-type span.place-features__values").text
                        except:
                            propiedad['tipo'] = None
                        try:
                            propiedad['tipo_vivienda'] = driver.find_element(By.CSS_SELECTOR, "span[data-test='property-type-value']").text
                        except:
                            propiedad['tipo_vivienda'] = None
                        
                        propiedad['categoria'] = propiedad['tipo_vivienda']
                        
                        try:
                            propiedad['precio'] = driver.find_element(By.CSS_SELECTOR, "div.prices-and-fees__price").text
                        except:
                            propiedad['precio'] = None
                        try:
                            propiedad['superficie'] = driver.find_element(By.CSS_SELECTOR, "div.details-item-value[data-test='area-value']").text
                        except:
                            propiedad['superficie'] = None
                        try:
                            propiedad['superficie_terreno'] = driver.find_element(By.CSS_SELECTOR, "span[data-test='plot-area-value']").text
                        except:
                            propiedad['superficie_terreno'] = None
                        try:
                            propiedad['habitaciones'] = driver.find_element(By.CSS_SELECTOR, "div.details-item-value[data-test='bedrooms-value']").text
                        except:
                            propiedad['habitaciones'] = None
                        try:
                            propiedad['banios'] = driver.find_element(By.CSS_SELECTOR, "div.details-item-value[data-test='full-bathrooms-value']").text
                        except:
                            propiedad['banios'] = None
                        try:
                            propiedad['caracteristica_propiedad'] = driver.find_element(By.CSS_SELECTOR, "div.facilities:nth-of-type(4)").text
                        except:
                            propiedad['caracteristica_propiedad'] = None
                        try:
                            propiedad['amenidades'] = driver.find_element(By.CSS_SELECTOR, "div.facilities:nth-of-type(5)").text
                        except:
                            propiedad['amenidades'] = None
                        try:
                            propiedad['caracteristicas'] = driver.find_element(By.CSS_SELECTOR, "div.place-details").text
                        except:
                            propiedad['caracteristicas'] = None
                        try:
                            propiedad['planta'] = driver.find_element(By.CSS_SELECTOR, ".floor span.place-features__values").text
                        except:
                            propiedad['planta'] = None
                        try:
                            propiedad['descripcion'] = driver.find_element(By.CSS_SELECTOR, "div#description-text").text
                        except:
                            propiedad['descripcion'] = None
                        try:
                            propiedad['fecha_publicacion'] = driver.find_element(By.CSS_SELECTOR, "div.date").text
                        except:
                            propiedad['fecha_publicacion'] = None
                        try:
                            propiedad['url_imagen'] = driver.find_element(By.CSS_SELECTOR, "div.swiper-wrapper img").get_attribute("src")
                        except:
                            propiedad['url_imagen'] = None
                        
                        propiedad['script_content'] = None
                        try:
                            if "mapData" in driver.page_source:
                                match = re.search(r'<script[^>]*>(.*?mapData.*?adLocationData.*?)</script>', driver.page_source, re.DOTALL)
                                if match:
                                    propiedad['script_content'] = match.group(1)
                        except:
                            pass
                        
                        propiedad['url'] = property_url
                        propiedad['fecha_consulta'] = time.strftime('%Y-%m-%d')

                        data_list.append(propiedad)
                        titulos_existentes.add(propiedad['titulo'])
                        propiedades_nuevas += 1
                        propiedades_nuevas_pagina += 1
                        print(f"      ✅ {idx+1}/30 OK ({propiedades_nuevas} total): {propiedad['titulo'][:30]}...")

                    except Exception as e:
                        propiedades_con_error += 1
                        propiedades_error_pagina += 1
                        print(f"      ❌ {idx+1}/30 ERROR: {str(e)[:45]}")
                        continue

                # RESUMEN DE PÁGINA
                print(f"\n   📊 RESUMEN P{paginas_procesadas}:")
                print(f"      • Links encontrados: {len(property_links)}")
                print(f"      • Nuevas descargadas: {propiedades_nuevas_pagina}")
                print(f"      • Repetidas saltadas: {propiedades_repetidas_pagina}")
                print(f"      • Con errores: {propiedades_error_pagina}")
                print(f"      • Procesadas total: {propiedades_nuevas_pagina + propiedades_repetidas_pagina + propiedades_error_pagina}")
                
                # Guardar incrementalmente
                if data_list:
                    data_rows = [[prop.get(col) for col in columnas] for prop in data_list]
                    df_nuevo = pd.DataFrame(data_rows, columns=columnas)
                    
                    mode = 'a' if os.path.exists(output_filename) else 'w'
                    header = not os.path.exists(output_filename)
                    df_nuevo.to_csv(output_filename, mode='a' if mode == 'a' else 'w', header=header if mode == 'w' else False, index=False, encoding="utf-8")
                    print(f"      💾 Guardadas en archivo\n")
                    data_list = []

            except Exception as e:
                print(f"Error de página: {str(e)}")
                break

    except Exception as e:
        print(f"Error crítico: {str(e)}")
    
    finally:
        driver.quit()

    print(f"\n{'='*80}")
    print(f"✅ DESCARGA COMPLETADA")
    print(f"{'='*80}")
    print(f"   ✅ Nuevas descargadas: {propiedades_nuevas}")
    print(f"   ⚠️  Repetidas saltadas: {propiedades_repetidas}")
    print(f"   ❌ Con errores: {propiedades_con_error}")
    print(f"   📊 Total procesadas: {propiedades_nuevas + propiedades_repetidas + propiedades_con_error}")
    print(f"{'='*80}\n")
    return propiedades_nuevas

In [3]:
# ⚙️ CONFIGURACIÓN DINÁMICA DE SCRAPING

ESTADOS = [
    "aguascalientes", "baja-california", "baja-california-sur", "campeche",
    "chiapas", "chihuahua", "coahuila-de-zaragoza", "colima", "distrito-federal",
    "durango", "mexico", "guanajuato", "guerrero", "hidalgo",
    "jalisco", "michoacan-de-ocampo", "morelos", "nayarit", "nuevo-leon", "oaxaca",
    "puebla", "queretaro-arteaga", "quintana-roo", "san-luis-potosi", "sinaloa",
    "sonora", "tabasco", "tamaulipas", "tlaxcala", "veracruz", "yucatan",
    "zacatecas"
]

TIPOS_PROPIEDAD = ['comercial', 'terreno', 'casa', 'departamento', 'offices']

def construir_url(estado, tipo_propiedad=None, accion='for-sale'):
    """
    Construye la URL de Lamudi dinámicamente.
    
    Args:
        estado (str): Estado de la URL (ej: 'aguascalientes')
        tipo_propiedad (str, optional): Tipo de propiedad (ej: 'comercial'). Si es None, obtiene todos.
        accion (str): Acción a realizar ('for-sale' o 'for-rent'). Default: 'for-sale'
    
    Returns:
        str: URL completa construida
    
    Ejemplos:
        >>> construir_url('aguascalientes')
        'https://www.lamudi.com.mx/aguascalientes/for-sale/'
        >>> construir_url('distrito-federal', 'comercial')
        'https://www.lamudi.com.mx/distrito-federal/comercial/for-sale/'
    """
    base_url = "https://www.lamudi.com.mx"
    
    if tipo_propiedad:
        url = f"{base_url}/{estado}/{tipo_propiedad}/{accion}/"
    else:
        url = f"{base_url}/{estado}/{accion}/"
    
    return url

def obtener_nombre_archivo(estado, tipo_propiedad=None, accion='for-sale'):
    """
    Genera nombre del archivo CSV basado en estado, tipo y acción.
    
    Args:
        estado (str): Estado
        tipo_propiedad (str, optional): Tipo de propiedad
        accion (str): Acción ('for-sale' o 'for-rent')
    
    Returns:
        str: Nombre del archivo CSV
    """
    if tipo_propiedad:
        filename = f"{estado}_{tipo_propiedad}_{accion.replace('-', '_')}.csv"
    else:
        filename = f"{estado}_{accion.replace('-', '_')}.csv"
    
    return filename

#### limpieza

In [ ]:

# 💾 FUNCIONES PARA GUARDAR Y REINTENTAR LINKS FALLIDOS

import json
from datetime import datetime

def guardar_links_fallidos(output_filename, failed_links):
    """
    Guarda los links que fallaron en un archivo JSON para reintentar después.
    
    Args:
        output_filename (str): Nombre base del CSV (ej: 'aguascalientes_casa.csv')
        failed_links (list): Lista de dicts con links fallidos
    """
    if not failed_links:
        print("✅ No hay links fallidos para guardar")
        return None
    
    # Nombre del archivo JSON basado en el CSV
    base_name = output_filename.replace('.csv', '')
    failed_filename = f"{base_name}_failed_links.json"
    
    # Guardar con timestamp
    datos_fallidos = {
        "timestamp": datetime.now().isoformat(),
        "total_fallidos": len(failed_links),
        "links": failed_links
    }
    
    with open(failed_filename, 'w', encoding='utf-8') as f:
        json.dump(datos_fallidos, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 LINKS FALLIDOS GUARDADOS:")
    print(f"   📁 Archivo: {failed_filename}")
    print(f"   📊 Total: {len(failed_links)} links fallidos")
    print(f"   ⏰ Timestamp: {datos_fallidos['timestamp']}")
    print(f"   🔍 Detalles por razón:")
    
    # Contar por razón de error
    razones = {}
    for link in failed_links:
        razon = link.get('razon', 'Unknown')
        razones[razon] = razones.get(razon, 0) + 1
    
    for razon, count in sorted(razones.items()):
        print(f"      • {razon}: {count} fallos")
    
    return failed_filename

def reintentar_links_fallidos(failed_filename, nuevo_csv=None):
    """
    Reintenta descargar los links que fallaron anteriormente.
    
    Args:
        failed_filename (str): Ruta del archivo JSON con links fallidos
        nuevo_csv (str, optional): Nombre del CSV para guardar nuevas propiedades
    
    Returns:
        list: Links que siguen fallando después de reintentar
    """
    print(f"\n🔄 REINTENTANDO LINKS FALLIDOS")
    print("=" * 80)
    
    # Cargar links fallidos
    with open(failed_filename, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    
    failed_links = datos.get('links', [])
    total_fallidos = len(failed_links)
    
    print(f"📂 Cargados {total_fallidos} links de: {failed_filename}")
    print(f"⏰ Original: {datos.get('timestamp', 'N/A')}")
    print("=" * 80)
    
    # Configurar driver
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.common.exceptions import TimeoutException
    import time
    
    chrome_options = Options()
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.set_page_load_timeout(20)  # Timeout más largo
    driver.implicitly_wait(5)
    wait = WebDriverWait(driver, 10)
    
    # Reintentar
    exitosos = 0
    seguidos_fallando = []
    
    try:
        for idx, link_data in enumerate(failed_links, 1):
            property_url = link_data.get('url')
            razon_original = link_data.get('razon', 'Unknown')
            pagina_original = link_data.get('pagina', '?')
            
            try:
                print(f"🔄 {idx}/{total_fallidos} Reintentando ({razon_original[:25]}...)...", end="")
                driver.get(property_url)
                
                # Verificar que cargó correctamente
                wait.until(EC.presence_of_element_located((By.CLASS_NAME, "ad-details")))
                
                print(" ✅ EXITOSO")
                exitosos += 1
                
            except TimeoutException:
                print(f" ⏱️  Timeout")
                seguidos_fallando.append({
                    **link_data,
                    "reintento": "timeout"
                })
            except Exception as e:
                print(f" ❌ {str(e)[:30]}")
                seguidos_fallando.append({
                    **link_data,
                    "reintento": str(e)[:60]
                })
    
    finally:
        driver.quit()
    
    # Resumen
    print("\n" + "=" * 80)
    print(f"📊 RESUMEN DE REINTENTOS:")
    print(f"   ✅ Exitosos: {exitosos}/{total_fallidos}")
    print(f"   ❌ Siguen fallando: {len(seguidos_fallando)}")
    print(f"   📈 Tasa éxito: {(exitosos/total_fallidos)*100:.1f}%")
    print("=" * 80)
    
    # Guardar los que siguen fallando
    if seguidos_fallando:
        new_failed_filename = failed_filename.replace('.json', '_reintento.json')
        with open(new_failed_filename, 'w', encoding='utf-8') as f:
            json.dump({
                "timestamp": datetime.now().isoformat(),
                "original_timestamp": datos.get('timestamp'),
                "exitosos": exitosos,
                "seguidos_fallando": len(seguidos_fallando),
                "links": seguidos_fallando
            }, f, ensure_ascii=False, indent=2)
        
        print(f"💾 Links que siguen fallando guardados en: {new_failed_filename}")
        return new_failed_filename
    
    return None


In [3]:
def limpiar_df(output_filename):

    df = pd.read_csv(output_filename)
    
    # Información sobre el dataset
    print(f"✅ Leyendo {len(df)} propiedades de {output_filename}")
    print(f"   Categorías presentes: {df['categoria'].value_counts().to_dict()}")
    
    # Eliminar el texto específico "Características de la propiedad\n" al principio de la columna
    df['caracteristica_propiedad'] = df['caracteristica_propiedad'].str.replace('^Características de la propiedad\n', '', regex=True)
    df['amenidades'] = df['amenidades'].str.replace('^Características del edificio\n', '', regex=True)
    df[['fecha_publicacion', 'publicado_por']] = df['fecha_publicacion'].str.split(' - Publicado por ', expand=True)

    ## Obtener datos textos

    # Aplica la transformación solo a los valores no nulos
    df['amenidades'] = df['amenidades'].apply(lambda x: unidecode(x).lower() if pd.notna(x) else x)

    # Crear la nueva columna
    df['estacionamiento'] = df['caracteristica_propiedad'].str.contains('estacionamiento', case=False, na=False).astype(int)

    # Primera parte: trabajar con 'caracteristica_propiedad'
    df['alberca'] = df['amenidades'].str.contains('alberca', case=False, na=False).astype(int)
    df['seguridad'] = df['amenidades'].str.contains('seguridad', case=False, na=False).astype(int)
    df['gimnasio'] = df['amenidades'].str.contains('gimnasio', case=False, na=False).astype(int)
    df['elevador'] = df['amenidades'].str.contains('elevador', case=False, na=False).astype(int)
    df['roof_garden'] = df['amenidades'].str.contains('roof', case=False, na=False).astype(int)
    df['jardin'] = df['amenidades'].str.contains('jardin', case=False, na=False).astype(int)
    df['salon'] = df['amenidades'].str.contains('salon', case=False, na=False).astype(int)
    df['mascotas'] = df['amenidades'].str.contains('pet', case=False, na=False).astype(int)
    
    # Segunda parte: si el valor es 0, tratar con 'descripcion'
    df['alberca'] = np.where(df['alberca'] == 0, 
                            df['descripcion'].str.contains('alberca', case=False, na=False).astype(int), 
                            df['alberca'])
    df['seguridad'] = np.where(df['amenidades'] == 0, 
                                df['descripcion'].str.contains('vigilancia', case=False, na=False).astype(int), 
                                df['seguridad'])

    df['seguridad'] = np.where(df['seguridad'] == 0, 
                                df['descripcion'].str.contains('seguridad', case=False, na=False).astype(int), 
                                df['seguridad'])
    df['gimnasio'] = np.where(df['gimnasio'] == 0, 
                            df['descripcion'].str.contains('gimnasio', case=False, na=False).astype(int), 
                            df['gimnasio'])
    df['elevador'] = np.where(df['elevador'] == 0, 
                            df['descripcion'].str.contains('elevador', case=False, na=False).astype(int), 
                            df['elevador'])
    df['roof_garden'] = np.where(df['roof_garden'] == 0, 
                                df['descripcion'].str.contains('roof', case=False, na=False).astype(int), 
                                df['roof_garden'])

    df['mascotas'] = np.where(df['mascotas'] == 0, 
                                df['descripcion'].str.contains('mascotas', case=False, na=False).astype(int), 
                                df['mascotas'])

    df['salon'] = np.where(df['salon'] == 0, 
                                df['descripcion'].str.contains('salon', case=False, na=False).astype(int), 
                                df['salon'])

    # Extraer coordenadas geográficas
    df['latitud'] = df['script_content'].str.extract(r'latitude:(.{9})', expand=False)
    df['latitud'] = df['latitud'].replace(r'[^0-9.-]', '', regex=True)
    df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')

    df['longitud'] = df['script_content'].str.extract(r'longitude:(.{10})', expand=False)
    df['longitud'] = df['longitud'].replace(r'[^0-9.-]', '', regex=True)
    df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')

    df['direccion'] = df['script_content'].str.extract(r'address:(.*?)\n', expand=False)
    df['direccion'] = df['direccion'].str.replace('`', '', regex=False)
    df['colonia'] = df['direccion'].str.split(',', n=2).str[1].str.strip()

    #### Convertir campos a números con manejo de propiedades no residenciales

    # Superficie (aplicable a todos)
    df['superficie'] = pd.to_numeric(df['superficie'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')
    
    # Superficie de terreno (especialmente para casas)
    if 'superficie_terreno' in df.columns:
        df['superficie_terreno'] = pd.to_numeric(df['superficie_terreno'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')

    # Precio (aplicable a todos)
    df['precio'] = pd.to_numeric(df['precio'].astype(str).str.replace(r'[^\d]', '', regex=True), errors='coerce')

    # Habitaciones (solo para residenciales, null para otros tipos)
    df['habitaciones'] = pd.to_numeric(df['habitaciones'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')

    # Baños (solo para residenciales, null para otros tipos)
    df['banios'] = pd.to_numeric(df['banios'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')

    ### Agregar CP 
    ruta_shp = "cp_cdmx/CP_09CDMX_v7"

    # Cargar el shapefile
    gdf = gpd.read_file(ruta_shp + ".shp")

    # Verificar si el shapefile se cargó correctamente
    if gdf.empty:
        print("El shapefile está vacío o no se pudo cargar correctamente.")
    else:
        # Verificar el CRS original
        print("CRS original de gdf:", gdf.crs)

        # Asignar el CRS original si no está definido (EPSG:6365)
        if gdf.crs is None:
            print("El shapefile no tiene CRS asignado. Asignando CRS EPSG:6365.")
            gdf = gdf.set_crs("EPSG:6365", allow_override=True)

        # Verificar que el CRS ha sido correctamente asignado antes de la transformación
        print("CRS después de asignar EPSG:6365:", gdf.crs)

        # Transformar el CRS a EPSG:4326 (WGS 84)
        gdf = gdf.to_crs("EPSG:4326")
        print("CRS después de la transformación:", gdf.crs)

    # Supongamos que 'df' ya contiene las columnas 'latitud' y 'longitud'
    # Convertimos 'df' en un GeoDataFrame con la geometría de los puntos
    geometry = [Point(lon, lat) for lon, lat in zip(df['longitud'], df['latitud'])]
    gdf_puntos = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # Asegúrate de usar el CRS correcto para tus puntos

    # Ahora, necesitamos verificar si los puntos en 'gdf_puntos' están dentro de alguna geometría de 'gdf'
    # Usamos el método 'sjoin' (spatial join) para combinar ambos GeoDataFrames
    df_con_cp = gpd.sjoin(gdf_puntos, gdf, how="left", predicate='within')

    # Ahora, 'df_con_cp' tendrá la columna 'd_cp' de 'gdf' y se llama 'cp' en df
    df_con_cp['cp'] = df_con_cp['d_cp']

    #### AGREGAR COLONIA

    ruta_shp = "coloniascdmx/colonias_iecm"

    # Cargar el shapefile
    gdf = gpd.read_file(ruta_shp + ".shp")

    # Verificar si el shapefile se cargó correctamente
    if gdf.empty:
        print("El shapefile está vacío o no se pudo cargar correctamente.")
    else:
        # Verificar el CRS original
        print("CRS original de gdf:", gdf.crs)

        # Asignar el CRS original si no está definido (EPSG:6365)
        if gdf.crs is None:
            print("El shapefile no tiene CRS asignado. Asignando CRS EPSG:6365.")
            gdf = gdf.set_crs("EPSG:6365", allow_override=True)

        # Verificar que el CRS ha sido correctamente asignado antes de la transformación
        print("CRS después de asignar EPSG:6365:", gdf.crs)

        # Transformar el CRS a EPSG:4326 (WGS 84)
        gdf = gdf.to_crs("EPSG:4326")
        print("CRS después de la transformación:", gdf.crs)

    # Supongamos que 'df' ya contiene las columnas 'latitud' y 'longitud'
    # Convertimos 'df' en un GeoDataFrame con la geometría de los puntos
    geometry = [Point(lon, lat) for lon, lat in zip(df['longitud'], df['latitud'])]
    gdf_puntos = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # Asegúrate de usar el CRS correcto para tus puntos

    # Ahora, necesitamos verificar si los puntos en 'gdf_puntos' están dentro de alguna geometría de 'gdf'
    # Usamos el método 'sjoin' (spatial join) para combinar ambos GeoDataFrames
    df_con_col = gpd.sjoin(gdf_puntos, gdf, how="left", predicate='within')

    # Ahora 'df_con_col' tiene las columnas de 'gdf' (incluyendo 'NOMUT')
    # La columna 'NOMUT' debe estar presente en 'df_con_col' después de la unión espacial
    df_con_col['colonia'] = df_con_col['NOMUT']  # Asignamos la columna 'NOMUT' a la columna 'colonia'

    df = df_con_col

    #### Corregir fecha 

    # Función para convertir fechas relativas en absolutas
    def convertir_fecha(fecha_str):
        ahora = datetime.now()
        
        # Si la fecha ya está en formato absoluto (e.g., "8 oct 2024")
        if re.search(r'\d{1,2} \w{3,} \d{4}', fecha_str):
            fecha_abs = pd.to_datetime(fecha_str, format='%d %b %Y', errors='coerce')
            return fecha_abs if pd.notna(fecha_abs) else fecha_str  # Si no se puede convertir, devolver original
        
        # Si la fecha es relativa (e.g., "Hace 1 semana, 2 días")
        match = re.findall(r'(\d+) (\w+)', fecha_str)
        if match:
            for cantidad, unidad in match:
                cantidad = int(cantidad)
                if 'hora' in unidad:
                    ahora -= timedelta(hours=cantidad)
                elif 'día' in unidad:
                    ahora -= timedelta(days=cantidad)
                elif 'semana' in unidad:
                    ahora -= timedelta(weeks=cantidad)
            return ahora

        return fecha_str  # Si no se pudo procesar, devolver original

    # Aplicar la conversión a la columna
    df['fecha_publicacion'] = df['fecha_publicacion'].apply(convertir_fecha)
    
    def convertir_fecha_formato(fecha):
        try:
            return parser.parse(str(fecha), dayfirst=True).strftime('%d/%m/%Y')
        except ValueError:
            return None  # Devuelve None si no puede parsear la fecha

    df['fecha_publicacion'] = df['fecha_publicacion'].apply(convertir_fecha_formato)

    #### Seleccionar columnas 

    df.drop(columns=['NOMUT', 'ID', 'CVEUT', 'DTTOLOC', 'index_right', 'geometry', 'script_content', 'caracteristicas', 'amenidades', 'caracteristica_propiedad'], inplace=True)

    print(f"✅ Limpieza completada. Dataset final: {len(df)} propiedades")
    
    return df

## Utilizar

In [4]:
# Funciones útiles para filtrar por categoría

def filtrar_por_categoria(df_limpio, categoria):
    """Filtra el dataset por categoría"""
    return df_limpio[df_limpio['categoria'] == categoria].copy()

def contar_propiedades_por_estado_y_tipo(transaccion='for-sale'):
    """
    Busca en Lamudi el número de propiedades por TIPO y por ESTADO.
    
    Args:
        transaccion (str): 'for-sale' (venta) o 'for-rent' (renta)
        
    Returns:
        pd.DataFrame: tabla con estados en filas, tipos de propiedad en columnas
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from webdriver_manager.chrome import ChromeDriverManager
    import time
    
    # Tipos de propiedad disponibles en Lamudi
    TIPOS_PROPIEDAD = ['comercial', 'terreno', 'casa', 'departamento', 'offices']
    
    # Lista de estados mexicanos
    ESTADOS = [
        "aguascalientes", "baja-california", "baja-california-sur", "campeche",
        "chiapas", "chihuahua", "coahuila-de-zaragoza", "colima", "distrito-federal",
        "durango", "mexico", "guanajuato", "guerrero", "hidalgo",
        "jalisco", "michoacan-de-ocampo", "morelos", "nayarit", "nuevo-leon", "oaxaca",
        "puebla", "queretaro-arteaga", "quintana-roo", "san-luis-potosi", "sinaloa",
        "sonora", "tabasco", "tamaulipas", "tlaxcala", "veracruz", "yucatan",
        "zacatecas"
    ]
    
    # Configuración de Chrome
    chrome_options = Options()
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.set_page_load_timeout(15)
    wait = WebDriverWait(driver, 10)
    
    # Diccionario para almacenar resultados
    resultados = {estado: {tipo: 0 for tipo in TIPOS_PROPIEDAD} for estado in ESTADOS}
    
    print(f"🔍 Contando propiedades por TIPO y ESTADO ({transaccion})...")
    print("=" * 80)
    
    total_requests = len(TIPOS_PROPIEDAD) * len(ESTADOS)
    request_actual = 0
    
    try:
        for tipo in TIPOS_PROPIEDAD:
            print(f"\n📦 Procesando tipo: {tipo.upper()}")
            print("-" * 80)
            
            for idx, estado in enumerate(ESTADOS, 1):
                request_actual += 1
                try:
                    # Construir URL: {estado}/{tipo}/{transaccion}/
                    url = f"https://www.lamudi.com.mx/{estado}/{tipo}/{transaccion}/"
                    driver.get(url)
                    time.sleep(0.8)
                    
                    # Extraer total de propiedades
                    try:
                        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "span[data-test='title-section-result-number']")))
                        total_text = driver.find_element(By.CSS_SELECTOR, "span[data-test='title-section-result-number']").text
                        total_text = total_text.replace(',', '').strip()
                        total = int(total_text)
                        resultados[estado][tipo] = total
                        pct = (request_actual / total_requests) * 100
                        print(f"  [{pct:>5.1f}%] {estado:25s}: {total:>6,d} ✓")
                        
                    except:
                        resultados[estado][tipo] = 0
                        pct = (request_actual / total_requests) * 100
                        print(f"  [{pct:>5.1f}%] {estado:25s}: sin resultados")
                    
                except Exception as e:
                    print(f"  [{estado:25s}]: Error ({str(e)[:25]})")
                    resultados[estado][tipo] = 0
        
        # Crear DataFrame con los resultados
        df = pd.DataFrame(resultados).T
        df.columns = [col.upper() for col in df.columns]
        df['TOTAL'] = df.sum(axis=1)
        
        # Mostrar resumen final
        print("\n" + "=" * 80)
        print("📊 RESUMEN POR ESTADO Y TIPO DE PROPIEDAD")
        print("=" * 80)
        print(df.to_string())
        
        # Totales por tipo
        print("\n" + "=" * 80)
        print("📈 TOTALES POR TIPO DE PROPIEDAD")
        print("=" * 80)
        totales_tipo = df.drop('TOTAL', axis=1).sum()
        for tipo, count in totales_tipo.items():
            print(f"  {tipo:15s}: {count:>10,d} propiedades")
        
        print("\n" + "=" * 80)
        print(f"✅ TOTAL GENERAL: {df['TOTAL'].sum():,d} propiedades en México")
        print(f"📈 Promedio por estado: {int(df['TOTAL'].mean()):,d} propiedades")
        estado_max = df['TOTAL'].idxmax()
        print(f"🏆 Estado con más propiedades: {estado_max} ({df.loc[estado_max, 'TOTAL']:.0f})")
        print("=" * 80)
        
        return df
        
    finally:
        driver.quit()

# Ejemplo de uso:
# df_venta = contar_propiedades_por_estado_y_tipo('for-sale')
# df_renta = contar_propiedades_por_estado_y_tipo('for-rent')

# df_renta.to_csv('resumen_estados/renta_lamudi.csv')
# df_venta.to_csv('resumen_estados/venta_lamudi.csv')

In [5]:
ESTADOS = [
    "aguascalientes", "baja-california", "baja-california-sur", "campeche",
    "chiapas", "chihuahua", "coahuila-de-zaragoza", "colima", "distrito-federal",
    "durango", "mexico", "guanajuato", "guerrero", "hidalgo",
    "jalisco", "michoacan-de-ocampo", "morelos", "nayarit", "nuevo-leon", "oaxaca",
    "puebla", "queretaro-arteaga", "quintana-roo", "san-luis-potosi", "sinaloa",
    "sonora", "tabasco", "tamaulipas", "tlaxcala", "veracruz", "yucatan",
    "zacatecas"
]
TIPOS_PROPIEDAD = ['terreno', 'casa', 'departamento', 'offices']

## 🏠 Descargar todos los tipos de Aguascalientes

## 🔄 Links Fallidos - Guardar y Reintentar

Después de descargar, los links que fallaron se guardan automáticamente en un archivo JSON. Puedes:

1. **Ver el reporte** de links fallidos
2. **Reintentar** descargar esos links más tarde
3. **Analizar** por qué fallaron

Ejemplo:
```python
# Guardar links fallidos manualmente
guardar_links_fallidos('aguascalientes_casa.csv', failed_links)

# Reintentar después
reintentar_links_fallidos('aguascalientes_casa_failed_links.json')
```

**Archivos generados:**
- `aguascalientes_casa_failed_links.json` - Links fallidos con detalles
- `aguascalientes_casa_failed_links_reintento.json` - Links que siguen fallando


In [7]:
# 📊 EJEMPLOS: Guardar y Reintentar Links Fallidos

# FUNCIÓN WRAPPER: Ejecuta scrape_lamudi y guarda automáticamente los links fallidos
def scrape_y_guardar_fallidos(start_url, output_filename):
    """
    Ejecuta scrape_lamudi y automáticamente guarda los links fallidos en JSON.
    
    Args:
        start_url (str): URL de inicio
        output_filename (str): Nombre del archivo CSV de salida
    """
    print(f"🌐 Iniciando descarga desde: {start_url}")
    
    # Ejecutar scraping
    global failed_links
    failed_links = []  # Inicializar lista global
    
    # Llamar a la función original
    scrape_lamudi(start_url, output_filename)
    
    # Guardar links fallidos si existen
    if failed_links:
        guardar_links_fallidos(output_filename, failed_links)
    else:
        print("✅ Se descargaron todas las propiedades sin errores")
    
    return failed_links

# EJEMPLO 1: Usar el wrapper que guarda automáticamente
# failed = scrape_y_guardar_fallidos('https://www.lamudi.com.mx/aguascalientes/casa/for-sale/', 
#                                     'aguascalientes_casa.csv')

# EJEMPLO 2: Reintentar links fallidos más tarde
# reintentar_links_fallidos('aguascalientes_casa_failed_links.json')

In [ ]:
# Descargar TODOS los tipos de propiedad para Aguascalientes
estado = "aguascalientes"
print(f"🔍 Descargando {len(TIPOS_PROPIEDAD)} tipos de propiedades para {estado.upper()}")
print("=" * 80)

stats_descarga = {}

for tipo in TIPOS_PROPIEDAD:
    print(f"\n📍 Descargando {tipo.upper()}...")
    
    # Construir URL
    start_url = f"https://www.lamudi.com.mx/{estado}/{tipo}/for-sale/"
    
    # Nombre del archivo
    output_filename = f'{estado}_{tipo}.csv'
    
    try:
        # Descargar datos CON almacenamiento automático de links fallidos
        failed = scrape_y_guardar_fallidos(start_url, output_filename)
        
        # Limpiar y guardar
        df_save = limpiar_df(output_filename)
        cleaned_filename = output_filename.replace('.csv', '_clean.csv')
        df_save.to_csv(cleaned_filename, index=False)
        
        stats_descarga[tipo] = {
            'propiedades': len(df_save),
            'links_fallidos': len(failed) if failed else 0,
            'archivo_raw': output_filename,
            'archivo_clean': cleaned_filename,
            'archivo_fallidos': f"{estado}_{tipo}_failed_links.json" if failed else None
        }
        
        print(f"✅ {tipo.upper()}: {len(df_save)} propiedades guardadas")
        if failed:
            print(f"   ⚠️  {len(failed)} links fallidos (guardados para reintentar)")
        print(f"   📁 Raw: {output_filename}")
        print(f"   📁 Clean: {cleaned_filename}")
        
    except Exception as e:
        print(f"❌ Error descargando {tipo}: {str(e)}")
        stats_descarga[tipo] = {'error': str(e)}

print("\n" + "=" * 80)
print(f"✅ DESCARGA COMPLETA DE {estado.upper()}")
print("=" * 80)

# Mostrar resumen
total_propiedades = sum(s.get('propiedades', 0) for s in stats_descarga.values())
total_fallidos = sum(s.get('links_fallidos', 0) for s in stats_descarga.values())

print(f"\n📊 RESUMEN POR TIPO:")
for tipo, stats in stats_descarga.items():
    if 'error' not in stats:
        print(f"   • {tipo:15s}: {stats['propiedades']:>4d} propiedades", end="")
        if stats['links_fallidos'] > 0:
            print(f" | ⚠️  {stats['links_fallidos']} fallidos")
        else:
            print()

print(f"\n📈 TOTALES:")
print(f"   Propiedades descargadas: {total_propiedades}")
if total_fallidos > 0:
    print(f"   Links fallidos: {total_fallidos}")
    print(f"   📝 Para reintentar usa: reintentar_links_fallidos('{estado}_TIPO_failed_links.json')")

print("=" * 80)

🔍 Descargando 4 tipos de propiedades para AGUASCALIENTES

📍 Descargando TERRENO...
🌐 Iniciando descarga desde: https://www.lamudi.com.mx/aguascalientes/terreno/for-sale/
📂 126 propiedades ya descargadas
📊 Total: 913 | Páginas estimadas: 31

🌐 P1: Cargando...
   ✅ Página cargada
   🔍 Buscando propiedades...
   📌 Links DOM encontrados: 30
   ✔️ Links válidos (con href): 30
   📋 Procesando 30 propiedades...
      ⚠️  1/30 REPETIDA: Terreno en venta - Condominio Resid...
      ⚠️  2/30 REPETIDA: TERRENOS RESIDENCIALES EN ANDREA AL...
      ⚠️  3/30 REPETIDA: TERRENO EDEN LOS SABINOS EN EXCELEN...
      ⚠️  4/30 REPETIDA: TERRENO EN VENTA EN RESIDENCIAL CAD...
      ⚠️  5/30 REPETIDA: EXCELENTE TERRENO EN VENTA EN CAMPE...
      ⚠️  6/30 REPETIDA: TERRENOS EN VENTA EN HACIENDA NUEVA...
      ⚠️  7/30 REPETIDA: EXCELENTE TERRENO EN VENTA EN RESER...
      ⚠️  8/30 REPETIDA: TERRENO EN VENTA EN MURALIA, AL NOR...
      ⚠️  9/30 REPETIDA: TERRENO EN VENTA EN RESERVA RESIDEN...
      ⚠️  10/30 

## 🔄 Tutorial: Reintentar Links Fallidos

### Paso 1: Identificar links fallidos ✅
Después de ejecutar la celda de descarga, se crean automáticamente archivos JSON con los links que fallaron:

```
aguascalientes_casa_failed_links.json
aguascalientes_terreno_failed_links.json
aguascalientes_departamento_failed_links.json
```

### Paso 2: Examinar el archivo (opcional) 📋
```python
import json

# Ver qué enlaces fallaron
with open('aguascalientes_casa_failed_links.json') as f:
    data = json.load(f)
    print(f"Total fallidos: {data['total_fallidos']}")
    for link in data['links'][:3]:  # Primeros 3
        print(f"  • {link['url'][:60]}... - {link['razon']}")
```

### Paso 3: Reintentar ⏱️
```python
# Reintentar los links fallidos
reintentar_links_fallidos('aguascalientes_casa_failed_links.json')
```

### Paso 4: Analizar resultados 📊
- Los que se descarguen exitosamente se eliminan de la lista
- Los que sigan fallando se guardan en: `aguascalientes_casa_failed_links_reintento.json`
- Puedes reintentar nuevamente si es necesario

### Resumen de razones de fallo comunes:
- ⏱️ **Timeout**: El sitio tarda en cargar, especialmente en conexiones lentas
- 🔌 **Conexión**: Error de conexión temporal
- 🚫 **HTTP**: Página no disponible temporalmente
- ⚠️ **Estructura HTML**: El sitio cambió su estructura

💡 **Tip**: Los timeouts suelen ser temporales. Reintentar después de algunas horas puede resolver el problema.

In [ ]:
# 🎯 EJEMPLO: Reintentar links fallidos

# OPCIÓN 1: Ver cuáles enlaces fallaron
print("📂 Archivos de links fallidos disponibles:")
import os
failed_files = [f for f in os.listdir('.') if 'failed_links.json' in f]
for f in failed_files:
    print(f"   • {f}")

print("\n" + "=" * 80)
print("❓ ¿No hay archivos de links fallidos? Eso significa ¡TODO SE DESCARGÓ EXITOSAMENTE! ✅")
print("=" * 80)

# OPCIÓN 2: Reintentar un archivo específico
print("\n💡 PARA REINTENTAR UN ARCHIVO ESPECÍFICO:")
print("""
   reintentar_links_fallidos('aguascalientes_casa_failed_links.json')
   reintentar_links_fallidos('aguascalientes_terreno_failed_links.json')
""")

# OPCIÓN 3: Reintentar todos los archivos fallidos
print("💡 PARA REINTENTAR TODOS:")
print("""
for archivo in failed_files:
    print(f"\\n🔄 Reintentando {archivo}...")
    reintentar_links_fallidos(archivo)
""")

# OPCIÓN 4: Analizar estadísticas de fallos
print("\n📊 PARA ANALIZAR ESTADÍSTICAS:")
print("""
import json

for archivo in failed_files:
    with open(archivo) as f:
        data = json.load(f)
    print(f"{archivo}: {data['total_fallidos']} fallos")
    
    # Agrupar por razón
    razones = {}
    for link in data['links']:
        razon = link['razon']
        razones[razon] = razones.get(razon, 0) + 1
    
    for razon, count in razones.items():
        print(f"   {razon}: {count}")
""")